In [ ]:
!pip install pypdf groq reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.3 MB/s eta 0:00:00


In [ ]:
from groq import Groq
from google.colab import userdata

client_groq = Groq(api_key=userdata.get('GROQ_API_KEY'))


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

def create_policy_pdf(filename):
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4

    # Title
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, height-50, "NORTHERN EUROPE BANK")
    c.drawString(50, height-70, "ESG REPORTING POLICY 2024")

    # Content
    c.setFont("Helvetica", 10)
    lines = [
        "",
        "Section 1 — Scope",
        "This policy applies to all corporate banking clients with annual",
        "turnover exceeding EUR 10 million. All clients must report ESG",
        "metrics annually.",
        "",
        "Section 2 — Reporting Requirements",
        "Clients must report the following metrics:",
        "- GHG Scope 1 and Scope 2 emissions in tonnes CO2 equivalent",
        "- Energy consumption in MWh",
        "- Water usage in cubic metres",
        "- Percentage of renewable energy used",
        "",
        "Section 3 — EU Taxonomy Alignment",
        "All clients in high-impact sectors must assess their activities",
        "against EU Taxonomy criteria. NACE codes must be provided for",
        "all reported activities. DNSH assessment is mandatory for",
        "High Risk sectors including Energy, Manufacturing and Transport.",
        "",
        "Section 4 — Deadlines",
        "Annual ESG reports must be submitted by 31 March each year.",
        "Non-compliant clients will be flagged for relationship review.",
        "",
        "Section 5 — Consequences of Non-Compliance",
        "Clients failing to submit reports face:",
        "- Credit facility review",
        "- Potential loan covenant breach",
        "- Escalation to senior relationship management",
    ]

    y = height - 100
    for line in lines:
        c.drawString(50, y, line)
        y -= 15

    c.save()
    print(f"✅ PDF created: {filename}")

create_policy_pdf("esg_policy.pdf")

✅ PDF created: esg_policy.pdf


In [ ]:
from pypdf import PdfReader

def read_pdf(filename):
    reader = PdfReader(filename)

    full_text = ""
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        full_text += text
        print(f"📄 Page {page_num + 1}: {len(text)} characters extracted")

    print(f"\n✅ Total text extracted: {len(full_text)} characters")
    return full_text

# Read our policy PDF
policy_text = read_pdf("esg_policy.pdf")
print("\n--- PREVIEW ---")
print(policy_text[:300])

📄 Page 1: 1062 characters extracted

✅ Total text extracted: 1062 characters

--- PREVIEW ---
NORTHERN EUROPE BANK
ESG REPORTING POLICY 2024
Section 1 — Scope
This policy applies to all corporate banking clients with annual
turnover exceeding EUR 10 million. All clients must report ESG
metrics annually.
Section 2 — Reporting Requirements
Clients must report the following metrics:
- GHG Scope


In [ ]:
def split_into_chunks(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # overlap keeps context between chunks

    print(f"✅ Document split into {len(chunks)} chunks")
    print(f"📦 Chunk size: {chunk_size} characters")
    print(f"🔗 Overlap: {overlap} characters")

    # Preview first chunk
    print(f"\n--- CHUNK 1 PREVIEW ---")
    print(chunks[0])

    return chunks

chunks = split_into_chunks(policy_text)

✅ Document split into 8 chunks
📦 Chunk size: 200 characters
🔗 Overlap: 50 characters

--- CHUNK 1 PREVIEW ---
NORTHERN EUROPE BANK
ESG REPORTING POLICY 2024
Section 1 — Scope
This policy applies to all corporate banking clients with annual
turnover exceeding EUR 10 million. All clients must report ESG
metrics


In [ ]:
def find_relevant_chunk(question, chunks):
    # Simple search — find chunk containing keywords from question
    question_words = question.lower().split()

    best_chunk = ""
    best_score = 0

    for chunk in chunks:
        chunk_lower = chunk.lower()
        score = sum(1 for word in question_words if word in chunk_lower)
        if score > best_score:
            best_score = score
            best_chunk = chunk

    return best_chunk

def ask_policy_question(question, chunks):
    # Step 1 — Find relevant chunk
    relevant_chunk = find_relevant_chunk(question, chunks)

    print(f"❓ Question: {question}")
    print(f"📦 Relevant chunk found: {relevant_chunk[:100]}...")

    # Step 2 — Send to AI with context
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a banking policy expert.
                Answer questions based ONLY on the provided policy document.
                If the answer is not in the document say so clearly."""
            },
            {
                "role": "user",
                "content": f"""Policy document excerpt:
                {relevant_chunk}

                Question: {question}"""
            }
        ]
    )

    answer = response.choices[0].message.content
    print(f"\n🤖 Answer: {answer}")
    return answer

# Test it!
ask_policy_question("What is the deadline for ESG reporting?", chunks)

❓ Question: What is the deadline for ESG reporting?
📦 Relevant chunk found: d for
all reported activities. DNSH assessment is mandatory for
High Risk sectors including Energy, ...

🤖 Answer: The deadline for submitting Annual ESG reports is 31 March.


'The deadline for submitting Annual ESG reports is 31 March.'

In [ ]:
questions = [
    "What happens if a client doesn't submit their ESG report?",
    "Which sectors require DNSH assessment?",
    "What metrics must clients report?",
    "What is the minimum turnover for ESG reporting?"
]

for question in questions:
    print("\n" + "="*50)
    ask_policy_question(question, chunks)


❓ Question: What happens if a client doesn't submit their ESG report?
📦 Relevant chunk found: 
Annual ESG reports must be submitted by 31 March each year.
Non-compliant clients will be flagged f...

🤖 Answer: According to the policy document, if a client doesn't submit their ESG report, they will be flagged for a relationship review, as stated under "Non-compliant clients will be flagged for relationship review." Additionally, Section 5 mentions "Consequences of Non-Compliance" for clients failing to submit reports, but the specific consequences are not detailed in the provided excerpt.

❓ Question: Which sectors require DNSH assessment?
📦 Relevant chunk found: ion 3 — EU Taxonomy Alignment
All clients in high-impact sectors must assess their activities
agains...

🤖 Answer: The policy document excerpt does not explicitly state which sectors require DNSH assessment. It mentions that DNSH assessment is related to clients in high-impact sectors, but it does not provide a clear answer to 

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a small but powerful embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Embedding model loaded")
print(f"📐 Embedding size: {embedding_model.get_sentence_embedding_dimension()} dimensions")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded
📐 Embedding size: 384 dimensions


/tmp/ipykernel_1705/4134027112.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"📐 Embedding size: {embedding_model.get_sentence_embedding_dimension()} dimensions")


In [ ]:
def embed_chunks(chunks):
    print("⏳ Creating embeddings for all chunks...")

    # Convert every chunk into a vector of numbers
    chunk_embeddings = embedding_model.encode(chunks)

    print(f"✅ Created {len(chunk_embeddings)} embeddings")
    print(f"📐 Each embedding: {len(chunk_embeddings[0])} dimensions")
    print(f"\n🔢 Preview of first embedding (first 5 numbers):")
    print(chunk_embeddings[0][:5])

    return chunk_embeddings

chunk_embeddings = embed_chunks(chunks)

⏳ Creating embeddings for all chunks...
✅ Created 8 embeddings
📐 Each embedding: 384 dimensions

🔢 Preview of first embedding (first 5 numbers):
[ 0.04843402  0.00103147  0.01609863 -0.05017535  0.05064397]


In [ ]:
def semantic_search(question, chunks, chunk_embeddings, top_k=1):
    # Convert question to embedding
    question_embedding = embedding_model.encode([question])

    # Calculate similarity between question and all chunks
    similarities = []
    for i, chunk_embedding in enumerate(chunk_embeddings):
        # Dot product — higher number = more similar meaning
        similarity = np.dot(question_embedding[0], chunk_embedding)
        similarities.append((i, similarity, chunks[i]))

    # Sort by similarity — highest first
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Return most relevant chunk
    best_index, best_score, best_chunk = similarities[0]

    print(f"🎯 Most relevant chunk (score: {best_score:.3f}):")
    print(f"{best_chunk[:150]}...")

    return best_chunk

# Test it
semantic_search(
    "What metrics must clients report?",
    chunks,
    chunk_embeddings
)

🎯 Most relevant chunk (score: 0.583):
UR 10 million. All clients must report ESG
metrics annually.
Section 2 — Reporting Requirements
Clients must report the following metrics:
- GHG Scope...


'UR 10 million. All clients must report ESG\nmetrics annually.\nSection 2 — Reporting Requirements\nClients must report the following metrics:\n- GHG Scope 1 and Scope 2 emissions in tonnes CO2 equivalent\n'

In [ ]:
def ask_policy_question_semantic(question, chunks, chunk_embeddings):
    print(f"❓ Question: {question}")

    # Step 1 — Semantic search for relevant chunk
    relevant_chunk = semantic_search(question, chunks, chunk_embeddings)

    # Step 2 — Send to AI with context
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a banking policy expert.
                Answer questions based ONLY on the provided policy document.
                If the answer is not in the document say so clearly."""
            },
            {
                "role": "user",
                "content": f"""Policy document excerpt:
                {relevant_chunk}

                Question: {question}"""
            }
        ]
    )

    answer = response.choices[0].message.content
    print(f"\n🤖 Answer: {answer}")
    print("=" * 50)
    return answer

# Test all 4 questions that failed yesterday
questions = [
    "What happens if a client doesn't submit their ESG report?",
    "Which sectors require DNSH assessment?",
    "What metrics must clients report?",
    "What is the minimum turnover for ESG reporting?"
]

for question in questions:
    ask_policy_question_semantic(question, chunks, chunk_embeddings)

❓ Question: What happens if a client doesn't submit their ESG report?
🎯 Most relevant chunk (score: 0.729):

Annual ESG reports must be submitted by 31 March each year.
Non-compliant clients will be flagged for relationship review.
Section 5 — Consequences o...

🤖 Answer: According to the policy document, if a client doesn't submit their ESG report, they will be flagged for relationship review, as stated under "Non-compliant clients will be flagged for relationship review." Additionally, the document mentions "Consequences of Non-Compliance" but does not specify further consequences in the provided excerpt.
❓ Question: Which sectors require DNSH assessment?
🎯 Most relevant chunk (score: 0.685):
d for
all reported activities. DNSH assessment is mandatory for
High Risk sectors including Energy, Manufacturing and Transport.
Section 4 — Deadlines...

🤖 Answer: According to the policy document, DNSH assessment is mandatory for High Risk sectors, which include:

1. Energy
2. Manufacturing
3.